In [36]:
# https://grok.com/share/bGVnYWN5_654f9fe9-1e40-445e-ba48-6e86d68645d6

import pandas as pd
import numpy as np
import datetime
from pandasql import sqldf
import sqlite3
import plotly.express as px


from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier  # Or another classifier
from sklearn.metrics import classification_report
from sklearn.preprocessing import OneHotEncoder  # Use OneHotEncoder




pd.options.display.max_rows = 50
pd.options.display.max_columns = 100

mysqldf = lambda q: sqldf(q, globals())

# conn = sqlite3.connect('optionsQuotes1.db')
# #c = conn.cursor()
# df = pd.read_sql('select * from data',conn)
# conn.close()
# df['time_converted'] = pd.to_datetime(df['time_converted'])

# conn.close()
df = pd.read_parquet('optionsDB2023-2025_deduped.parquet')
df['time_converted'] = pd.to_datetime(df['time_converted'])

In [ ]:
# daily_summary  = mysqldf(""" 
#              SELECT
# DATE(Time_Converted) AS 'Date',
# MAX(CASE WHEN TYPE = 'call' THEN options_pct_change ELSE NULL END) AS 'Call_Max_Daily',
# MAX(CASE WHEN TYPE = 'put' THEN options_pct_change ELSE NULL END) AS 'Put_Max_Daily'
# FROM df
# WHERE DTE_ADJUSTED = 0 AND options_earliest_open>0.2
# GROUP BY DATE(Time_Converted)
#         """)

daily_summary_sql  = mysqldf(""" 
    SELECT 
    o.Date,
    o.Call_Max_Daily,
    o.Put_Max_Daily,
    e.equity_open,
    e.close_equity,
    e.high_equity,
    e.low_equity
FROM (
    SELECT 
        DATE(Time_Converted) AS 'Date',
        MAX(CASE WHEN TYPE = 'call' THEN options_pct_change ELSE NULL END) AS Call_Max_Daily,
        MAX(CASE WHEN TYPE = 'put' THEN options_pct_change ELSE NULL END) AS Put_Max_Daily
    FROM df
    WHERE DTE_ADJUSTED = 0 
        AND options_earliest_open > 0.2
        AND ticker = 'SPY'
    GROUP BY DATE(Time_Converted)
) AS o
LEFT JOIN (
    SELECT
        daily_agg.Date,
        daily_agg.equity_open,
        daily_agg.high_equity,
        daily_agg.low_equity,
        last_candle.close_equity
    FROM (
        SELECT
            DATE(time_converted) AS Date,
            MAX(equity_start_price) AS equity_open,
            MAX(high_equity) AS high_equity,
            MIN(low_equity) AS low_equity
        FROM df
        WHERE ticker = 'SPY'
        GROUP BY DATE(time_converted)
    ) AS daily_agg
    LEFT JOIN (
        SELECT
            DATE(time_converted) AS Date,
            close_equity
        FROM (
            SELECT
                *,
                ROW_NUMBER() OVER (PARTITION BY DATE(time_converted) ORDER BY TIME(time_converted) DESC) AS rn
            FROM df
            WHERE ticker = 'SPY'
                AND close_equity IS NOT NULL
                AND TIME(time_converted) <= '15:45:00'
        ) AS ranked_data
        WHERE rn = 1
    ) AS last_candle ON daily_agg.Date = last_candle.Date
) AS e ON o.Date = e.Date
        """)


daily_summary_sql['Max'] = daily_summary_sql[['Call_Max_Daily', 'Put_Max_Daily']].max(axis=1)
daily_summary_sql['Date'] = pd.to_datetime(daily_summary_sql['Date'])
daily_summary_sql['weekday'] = daily_summary_sql['Date'].dt.day_name()


In [37]:
def create_daily_summary(df):
    # Convert time_converted to datetime
    df['time_converted'] = pd.to_datetime(df['time_converted'])
    
    # Step 1: Process options data
    filtered_options = df[(df['ticker'] == 'SPY') & 
                         (df['DTE_adjusted'] == 0) & 
                         (df['options_earliest_open'] > 0.2)]
    
    options_summary = (filtered_options.groupby(filtered_options['time_converted'].dt.date)
                      .agg({
                          'options_pct_change': [
                              ('Call_Max_Daily', lambda x: x[filtered_options.loc[x.index, 'type'] == 'call'].max()),
                              ('Put_Max_Daily', lambda x: x[filtered_options.loc[x.index, 'type'] == 'put'].max())
                          ]
                      })
                      .reset_index()
                      .rename(columns={'time_converted': 'Date'}))
    
    options_summary.columns = ['Date', 'Call_Max_Daily', 'Put_Max_Daily']
    options_summary['Date'] = pd.to_datetime(options_summary['Date']).dt.date  # Keep as date only
    
    # Step 2: Process equity data
    filtered_equity = df[df['ticker'] == 'SPY'].copy()
    
    # Daily aggregates
    daily_agg = (filtered_equity.groupby(filtered_equity['time_converted'].dt.date)
                 .agg({
                     'equity_start_price': 'max',  # equity_open
                     'high_equity': 'max',
                     'low_equity': 'min'
                 })
                 .reset_index()
                 .rename(columns={'time_converted': 'Date', 'equity_start_price': 'equity_open'}))
    
    daily_agg['Date'] = pd.to_datetime(daily_agg['Date']).dt.date  # Keep as date only
    
    # Last close_equity before 3:45 PM with fallback to last non-null
    filtered_equity['time_only'] = filtered_equity['time_converted'].dt.time
    # Get all SPY data first, then filter and sort
    equity_by_day = filtered_equity[filtered_equity['ticker'] == 'SPY'].copy()
    last_candle_candidates = equity_by_day[equity_by_day['close_equity'].notna()]  # Only rows with valid close_equity
    last_candle_candidates['time_only'] = last_candle_candidates['time_converted'].dt.time  # Add time_only here
    if last_candle_candidates.empty:
        print("Warning: No valid close_equity values found. Check data.")
    else:
        # Sort globally by date ascending and time descending, then filter and sort within groups
        last_candle = (last_candle_candidates
                      .sort_values(by=['time_converted'], key=lambda x: x.map(lambda t: (t.date(), -t.hour * 3600 - t.minute * 60 - t.second)), ascending=True)
                      .groupby(last_candle_candidates['time_converted'].dt.date)
                      .apply(lambda x: x[x['time_only'] <= pd.to_datetime('15:45:00').time()].sort_values(by='time_converted', ascending=False).head(1))
                      .reset_index(drop=True)
                      [['time_converted', 'close_equity']]
                      .rename(columns={'time_converted': 'Date'}))
        last_candle['Date'] = pd.to_datetime(last_candle['Date']).dt.date  # Keep as date only
        print(f"Last candle candidates rows: {len(last_candle_candidates)}")
        print(f"Unique last candle rows: {len(last_candle)}")
        print(last_candle.head())  # Debug to check content
    
    # Join daily_agg with last_candle
    equity_summary = daily_agg.merge(last_candle, on='Date', how='left')
    
    # Step 3: Left join options and equity data
    daily_summary = options_summary.merge(equity_summary, on='Date', how='left')
    
    # Ensure Date is in datetime format for final output
    daily_summary['Date'] = pd.to_datetime(daily_summary['Date'])
    
    # Debug: Print intermediate and final results
    print(f"Options rows: {len(options_summary)}")
    print(f"Equity rows: {len(equity_summary)}")
    print(f"Final daily_summary rows: {len(daily_summary)}")
    print(daily_summary.head())
    print(daily_summary.dtypes)
    print(f"NaN count in close_equity: {daily_summary['close_equity'].isna().sum()}")
    
    return daily_summary
daily_summary = create_daily_summary(df)
daily_summary['Max'] = daily_summary[['Call_Max_Daily', 'Put_Max_Daily']].max(axis=1)
daily_summary['Date'] = pd.to_datetime(daily_summary['Date'])
daily_summary['weekday'] = daily_summary['Date'].dt.day_name()

C:\Users\dkarl\AppData\Local\Temp\ipykernel_31124\107800885.py:43: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  last_candle_candidates['time_only'] = last_candle_candidates['time_converted'].dt.time  # Add time_only here


Last candle candidates rows: 2133891
Unique last candle rows: 611
         Date  close_equity
0  2023-01-03      380.8400
1  2023-01-04      383.7400
2  2023-01-05      379.4000
3  2023-01-06      388.0300
4  2023-01-09      387.8657
Options rows: 611
Equity rows: 611
Final daily_summary rows: 611
        Date  Call_Max_Daily  Put_Max_Daily  equity_open  high_equity  \
0 2023-01-03        1.000000      10.880000       385.29       385.40   
1 2023-01-04        2.363636       1.419355       382.63       385.88   
2 2023-01-05        1.411765       1.078755       379.71       381.84   
3 2023-01-06       13.225806       1.000000       380.21       389.25   
4 2023-01-09        2.790698       1.705069       390.49       393.70   

   low_equity  close_equity  
0    377.8310      380.8400  
1    380.0000      383.7400  
2    378.7600      379.4000  
3    379.4127      388.0300  
4    387.6700      387.8657  
Date              datetime64[ns]
Call_Max_Daily           float64
Put_Max_Daily   

In [3]:
daily_summary.head(3)

,Date,Call_Max_Daily,Put_Max_Daily,equity_open,high_equity,low_equity,close_equity,Max,weekday
0,2023-01-03,1.000000,10.880000,385.29,385.40,377.831,380.84,10.880000,Tuesday
1,2023-01-04,2.363636,1.419355,382.63,385.88,380.000,383.74,2.363636,Wednesday
2,2023-01-05,1.411765,1.078755,379.71,381.84,378.760,379.40,1.411765,Thursday


In [38]:
## Do not run. Old feature engineering code

# But somehow it is better than the new one
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import GridSearchCV

import numpy as np

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

def day_classification(row):
    very_high_range = 4
    high_range = 2
    average_range = 1.5
    if row['Max'] > very_high_range:
        return 'very_high'
    elif row['Max'] > high_range:
        return 'high'
    elif row['Max'] > average_range:
        return 'average'
    else:
        return 'low'

# Assume daily_summary has 'Max' from SQL
daily_summary['day_classification'] = daily_summary.apply(day_classification, axis=1)
def create_lags(df):
    df['d1'] = df.loc[:,'day_classification'].shift(1)
    df['d2'] = df.loc[:,'day_classification'].shift(2)
    df['d3'] = df.loc[:,'day_classification'].shift(3)
    df['weekday_encoded'] = LabelEncoder().fit_transform(df['weekday'])  # Encode weekday
    ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    weekday_encoded = ohe.fit_transform(df[['weekday']])
    df = pd.concat([df, pd.DataFrame(weekday_encoded, columns=ohe.get_feature_names_out(['weekday']))], axis=1)
    df['week_number_in_month'] = df['Date'].apply(lambda x: (x.day-1) // 7 +1 )  # Week number in month
    df['week_sin'] = np.sin(2 * np.pi * df['week_number_in_month'] / 5)  # Assuming max 5 weeks
    df['week_cos'] = np.cos(2 * np.pi * df['week_number_in_month'] / 5)
    df['month_half'] = df['Date'].apply(lambda x: 1 if x.day <= 15 else 2)  # First or second half of the month
    df.dropna(inplace=True)
    return df
def split_feature(df):
    # X = df.loc[:,['d1', 'd2', 'd3', 'weekday_encoded', 'week_sin', 'week_cos']]  # Features
    y = df.loc[:,'day_classification']
    # features_list
    features_list = ['d1', 'week_sin']
    weekdays = [col for col in df.columns if 'weekday_' in col]
    # features_list.extend(weekdays)
    X = df.loc[:,features_list]
    #
    # agggregate days in 2 groups: <2 and >=2

    y = y.apply(lambda x: 1 if 'high' in x else 0)
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=14, stratify=y) # Stratify for class balance
    return X_train, X_test, y_train, y_test
df_work = create_lags(daily_summary)
X_train, X_test, y_train, y_test = split_feature(df_work)

# le = LabelEncoder()
# y_train_encoded = le.fit_transform(y_train)
# y_test_encoded = le.transform(y_test)

# Encode the features (d1, d2, d3)
le_features = LabelEncoder()
# for column in ['d1', 'd2', 'd3']:
for column in ['d1']:
    X_train[column] = le_features.fit_transform(X_train[column])
    X_test[column] = le_features.transform(X_test[column])
# Train a classifier (RandomForest is a good starting point)
rf = RandomForestClassifier(random_state=12, class_weight='balanced') # You can try other models (e.g., Gradient Boosting)
grid_search = GridSearchCV(rf, param_grid, cv=5, n_jobs=-1, scoring='balanced_accuracy', verbose=2)
grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
model = grid_search.best_estimator_

# Make predictions on the test set
y_pred = model.predict(X_test)

# Evaluate the model (convert predictions back to original labels for reporting)
#y_pred_labels = le.inverse_transform(y_pred)
print("Classification Report:\n", classification_report(y_test, y_pred))

# Evaluate the model
#print(classification_report(y_test, y_pred))

# Feature Importance (to see which lags are most influential)
feature_importances = model.feature_importances_
print("Feature Importances:", feature_importances)


features = X_train.columns.tolist()

Fitting 5 folds for each of 81 candidates, totalling 405 fits
Best parameters: {'max_depth': 10, 'min_samples_leaf': 1, 'min_samples_split': 10, 'n_estimators': 200}
Classification Report:
               precision    recall  f1-score   support

           0       0.30      0.45      0.36        40
           1       0.65      0.49      0.56        82

    accuracy                           0.48       122
   macro avg       0.47      0.47      0.46       122
weighted avg       0.53      0.48      0.49       122

Feature Importances: [0.38493037 0.61506963]


In [ ]:
# Optional: compare daily_summary with daily_summary_sql as a sanity check

# Optional: Align indexes
daily_summary = daily_summary.reset_index(drop=True)
daily_summary_sql = daily_summary_sql.reset_index(drop=True)

# Optional: Check column match
assert set(daily_summary.columns) == set(daily_summary_sql.columns), "DataFrames have different columns"

# Compare
for column in daily_summary.columns:
    if not daily_summary[column].equals(daily_summary_sql[column]):
        print(f"Column '{column}' does not match between daily_summary and daily_summary_sql.")
        print("Differences:")
        mask = daily_summary[column] != daily_summary_sql[column]
        print(daily_summary[column][mask])
        print(daily_summary_sql[column][mask])
    else:
        print(f"Column '{column}' matches between daily_summary and daily_summary_sql.")


In [ ]:
## DO NOT RUN THIS CELL

# Add lagged features to daily_summary
def create_lags(df, lags=[1, 3]):
    df = df.sort_values('Date')  # Ensure chronological order
    for lag in lags:
        df[f'close_equity_lag{lag}'] = df['close_equity'].shift(lag)
        df[f'Max_Daily_lag{lag}'] = df['Max'].shift(lag)
    return df.dropna()  # Drop rows with NaN from initial lags

# Apply lagging
daily_summary_with_lags = create_lags(daily_summary)

# Debug
print(daily_summary_with_lags.head())
print(f"Rows after lagging: {len(daily_summary_with_lags)}")

        Date  Call_Max_Daily  Put_Max_Daily  equity_open  high_equity  \
3 2023-01-06       13.225806       1.000000       380.21      389.250   
4 2023-01-09        2.790698       1.705069       390.49      393.700   
5 2023-01-10        1.480583       1.593750       388.42      390.650   
6 2023-01-11        1.460000       1.535714       393.00      395.600   
7 2023-01-12        2.956522       1.000000       393.36      398.485   

   low_equity  close_equity        Max    weekday  close_equity_lag1  \
3    379.4127      388.0300  13.225806     Friday           379.4000   
4    387.6700      387.8657   2.790698     Monday           388.0300   
5    386.2700      390.6300   1.593750    Tuesday           387.8657   
6    391.3800      395.5000   1.535714  Wednesday           390.6300   
7    392.4200      396.9400   2.956522   Thursday           395.5000   

   Max_Daily_lag1  close_equity_lag3  Max_Daily_lag3  
3        1.411765           380.8400       10.880000  
4       13.225806 

In [44]:
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, recall_score, make_scorer
from sklearn.preprocessing import LabelBinarizer
from sklearn.model_selection import StratifiedKFold



# Define hyperparameters for RandomForestClassifier
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# Define classification ranges for day_classification
very_high_range = 4
high_range = 2
average_range = 1.5

def day_classification(row):
    if row['Max'] > very_high_range:
        return 'very_high'
    elif row['Max'] > high_range:
        return 'high'
    elif row['Max'] > average_range:
        return 'average'
    else:
        return 'low'

# Add initial features to daily_summary
daily_summary['spy_direction_lag1'] = (daily_summary['close_equity'] - daily_summary['equity_open']).shift(1)
daily_summary['intraday_range_lag1'] = (daily_summary['high_equity'] - daily_summary['low_equity']).shift(1)
daily_summary['spy_direction_lag2'] = (daily_summary['close_equity'] - daily_summary['equity_open']).shift(2)
daily_summary['day_classification'] = daily_summary.apply(day_classification, axis=1)

def create_lags(df):
    df['d1'] = df.loc[:, 'day_classification'].shift(1)
    df['d2'] = df.loc[:, 'day_classification'].shift(2)
    df['d3'] = df.loc[:, 'day_classification'].shift(3)
    df['d4'] = df.loc[:, 'day_classification'].shift(4)
    df['d5'] = df.loc[:, 'day_classification'].shift(5)
    df['weekday_encoded'] = LabelEncoder().fit_transform(df['weekday'])
    ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    weekday_encoded = ohe.fit_transform(df[['weekday']])
    df = pd.concat([df, pd.DataFrame(weekday_encoded, columns=ohe.get_feature_names_out(['weekday']))], axis=1)
    df['week_number_in_month'] = df['Date'].apply(lambda x: (x.day - 1) // 7 + 1)
    df['week_sin'] = np.sin(2 * np.pi * df['week_number_in_month'] / 5)
    df['week_cos'] = np.cos(2 * np.pi * df['week_number_in_month'] / 5)
    df['max_lag1'] = df['Max'].shift(1)
    df['max_lag2'] = df['Max'].shift(2)
    df['max_lag3'] = df['Max'].shift(3)
    df['spy_direction_lag1'] = df['spy_direction_lag1']
    df['intraday_range_lag1'] = df['intraday_range_lag1']
    df['spy_direction_lag2'] = df['spy_direction_lag2']
    df.dropna(inplace=True)
    return df

def split_feature(df):
    features_list = ['d1', 'd2', 'd3', 'd4', 'd5', 'week_sin', 'week_cos', 'max_lag1', 'max_lag2', 'max_lag3', 'spy_direction_lag1', 'intraday_range_lag1', 'spy_direction_lag2']
    weekdays = [col for col in df.columns if 'weekday_' in col]
    features_list = ['d1','d2','d3','week_sin', 'weekday_encoded']  # Simplified feature set for testing
    # features_list.extend(weekdays)
    X = df.loc[:, features_list]
    y = df.apply(lambda x: 1 if x['Max'] > 2 else 0, axis=1)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=14, stratify=y)
    return X_train, X_test, y_train, y_test

# Create features and split data
df_work = create_lags(daily_summary)
X_train, X_test, y_train, y_test = split_feature(df_work)

# Reset index to positional integers
X_test_original_indices = X_test.index
X_train = X_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)


# Encode categorical features
le_features = LabelEncoder()

for column in ['d1', 'd2','d3']: #, 'd4', 'd5']:
    X_train[column] = le_features.fit_transform(X_train[column])
    X_test[column] = X_test[column].map(lambda s: le_features.transform([s])[0] if s in le_features.classes_ else -1)

# Custom function to debug folds
def print_fold_distribution(cv, X, y):
    fold_count = 0
    for train_idx, test_idx in cv.split(X, y):
        print(f"\nFold {fold_count}:")
        y_train_fold = y.iloc[train_idx]  # Use iloc for positional indexing
        y_test_fold = y.iloc[test_idx]
        print(f"  y_train value counts: {y_train_fold.value_counts()}")
        print(f"  y_test value counts: {y_test_fold.value_counts()}")
        fold_count += 1
    print(f"\nTotal folds debugged: {fold_count}")

# Train and tune the volatility model with fold debugging
rf = RandomForestClassifier(random_state=12, class_weight='balanced')
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=14)
# print_fold_distribution(cv, X_train, y_train)  # Debug fold distributions before fitting
grid_search = GridSearchCV(rf, param_grid, cv=5, n_jobs=-1, scoring='balanced_accuracy', verbose=2)
grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
model = grid_search.best_estimator_

# Evaluate the volatility model
y_pred = model.predict(X_test)
print("Classification Report:\n", classification_report(y_test, y_pred))

# Feature Importance
feature_importances = model.feature_importances_
print("Feature Importances:", dict(zip(X_train.columns, feature_importances)))




Fitting 5 folds for each of 81 candidates, totalling 405 fits
Best parameters: {'max_depth': 20, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 200}
Classification Report:
               precision    recall  f1-score   support

           0       0.42      0.33      0.37        40
           1       0.70      0.78      0.74        82

    accuracy                           0.63       122
   macro avg       0.56      0.55      0.55       122
weighted avg       0.61      0.63      0.62       122

Feature Importances: {'d1': np.float64(0.1969275031883192), 'd2': np.float64(0.17321157077963284), 'd3': np.float64(0.17582653217199265), 'week_sin': np.float64(0.21638796646016273), 'weekday_encoded': np.float64(0.23764642739989267)}


In [40]:
X_train.head(10)

,d1,week_sin,weekday_encoded
0,0,9.510565e-01,2
1,1,-5.877853e-01,0
2,1,9.510565e-01,4
3,3,-5.877853e-01,4
4,1,5.877853e-01,3
5,1,-2.449294e-16,3
6,2,-9.510565e-01,3
7,1,-9.510565e-01,2
8,3,9.510565e-01,4
9,2,5.877853e-01,0


In [35]:
daily_summary.shape

(611, 19)

In [ ]:
# Calculate EV with Max-based profit logic
test_df = X_test.copy()
test_df['y_true'] = y_test
test_df['y_pred'] = y_pred
test_df['Max'] = df.loc[X_test_original_indices]['Max'].values

# Define profit/loss logic based on Max exceeding target
initial_investment = 200  # $200 per trade
max_profit_multiple = 3  # Cap profit target at 3x
take_profit = min(initial_investment * max_profit_multiple, initial_investment * 15)  # Cap at 3x or input DK: stupid line from Grok
profit_target = take_profit / initial_investment  # Target multiple (e.g., 3)
test_df['actual_profit'] = test_df.apply(
    lambda row: take_profit * 0.5 if (row['y_pred'] == 1 and row['y_pred'] == row['y_true'] and row['Max'] >= profit_target) else 
                -initial_investment if (row['y_pred'] == 1 and (row['y_pred'] != row['y_true'] or row['Max'] < profit_target)) else 0, axis=1)

# Aggregate results
total_trades = test_df['y_pred'].sum()  # Total predicted Max > 2 days
true_positives = test_df[(test_df['y_pred'] == 1) & (test_df['y_pred'] == test_df['y_true'])].shape[0]
profitable_trades = test_df[(test_df['actual_profit'] > 0)].shape[0]  # Count all profitable trades
false_positives = test_df[(test_df['y_pred'] == 1) & (test_df['y_pred'] != test_df['y_true'])].shape[0]
missed_targets = test_df[(test_df['y_pred'] == 1) & (test_df['y_pred'] == test_df['y_true']) & (test_df['Max'] < profit_target)].shape[0]

# Calculate EV
total_profit = test_df['actual_profit'].sum()
ev_per_trade = total_profit / total_trades if total_trades > 0 else 0
ev_per_day = total_profit / len(daily_summary)  # Average profit per day across full dataset

print(f"Total Trades (Max > 2 predicted): {total_trades}")
print(f"True Positives: {true_positives}")
print(f"Profitable Trades: {profitable_trades}")
print(f"False Positives: {false_positives}")
print(f"Missed Targets (True Positives with Max < Target): {missed_targets}")
print(f"Total Profit: ${total_profit:.2f}")
print(f"EV per Trade: ${ev_per_trade:.2f}")
print(f"EV per Day (full dataset): ${ev_per_day:.2f}")

Total Trades (Max > 2 predicted): 115
True Positives: 78
Profitable Trades: 45
False Positives: 37
Missed Targets (True Positives with Max < Target): 33
Total Profit: $-500.00
EV per Trade: $-4.35
EV per Day (full dataset): $-0.82


In [52]:
# Prepare data for chart
profit_targets = np.arange(1.5, 8.1, 0.3)  # From 1.5 to 8 in 0.3 increments
ev_values = []

for target in profit_targets:
    take_profit = initial_investment * target
    take_profit = min(initial_investment * max_profit_multiple, take_profit)  # Cap at 3x
    profit_target = take_profit / initial_investment
    temp_profit = test_df.apply(
        lambda row: take_profit * 0.5 if (row['y_pred'] == 1 and row['y_pred'] == row['y_true'] and row['Max'] >= profit_target) else 
                    -initial_investment if (row['y_pred'] == 1 and (row['y_pred'] != row['y_true'] or row['Max'] < profit_target)) else 0, axis=1)
    total_profit = temp_profit.sum()
    ev = total_profit / total_trades if total_trades > 0 else 0
    ev_values.append(ev)

# Create DataFrame for Plotly Express
chart_data = pd.DataFrame({
    'Profit Target (x Initial Investment)': profit_targets,
    'EV per Trade ($)': ev_values
})

# Create Plotly Express line chart
fig = px.line(chart_data, x='Profit Target (x Initial Investment)', y='EV per Trade ($)',
              title='EV per Trade vs Profit Target',
              markers=True,  # Add markers at data points
              line_shape='linear')

# Update layout
fig.update_layout(
    xaxis=dict(tickmode='array', tickvals=np.arange(1.5, 8.1, 1)),
    yaxis=dict(range=[min(ev_values) - 50 if ev_values else -100, max(ev_values) + 50 if ev_values else 100]),
    xaxis_title='Profit Target (x Initial Investment)',
    yaxis_title='EV per Trade ($)'
)

# Show the plot
fig.show()

In [56]:
daily_summary.to_csv('daily_summary.csv', index=False)

In [ ]:
daily_summary.groupby(['day_classification','weekday'])

day_classification  weekday    Date        Call_Max_Daily  Put_Max_Daily  equity_open  high_equity  low_equity  close_equity  Max        spy_direction_lag1  intraday_range_lag1  spy_direction_lag2  d1         d2         d3         d4         d5         weekday_encoded
average             Friday     2023-02-10  1.556962        1.114846       405.8300     408.4400     405.01      408.03        1.556962   -6.220              8.6000               -3.595              very_high  very_high  high       low        high       0                  1
                               2023-02-24  1.601594        1.122807       394.8000     397.2500     393.64      396.39        1.601594   -0.420              5.9500               -1.490              very_high  average    very_high  low        high       0                  1
                               2023-04-21  1.122881        1.909091       411.9700     412.6800     410.17      412.18        1.909091   -0.170              3.4300                1.86

In [24]:
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import GridSearchCV

import numpy as np

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}


def create_lags(df):
    df['d1'] = df.loc[:,'day_classification'].shift(1)
    df['d2'] = df.loc[:,'day_classification'].shift(2)
    df['d3'] = df.loc[:,'day_classification'].shift(3)
    df['weekday_encoded'] = LabelEncoder().fit_transform(df['weekday'])  # Encode weekday
    df['week_number_in_month'] = df['Date'].apply(lambda x: (x.day-1) // 7 +1 )  # Week number in month
    df.dropna(inplace=True)
    return df
def split_feature(df):
    X = df.loc[:,['d1', 'd2', 'd3', 'weekday_encoded', 'week_number_in_month']]
    y = df.loc[:,'day_classification']

    # agggregate days in 2 groups: <2 and >=2

    y = y.apply(lambda x: 1 if 'high' in x else 0)
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y) # Stratify for class balance
    return X_train, X_test, y_train, y_test
df = create_lags(daily_summary)
X_train, X_test, y_train, y_test = split_feature(df)

# le = LabelEncoder()
# y_train_encoded = le.fit_transform(y_train)
# y_test_encoded = le.transform(y_test)

# Encode the features (d1, d2, d3)
le_features = LabelEncoder()
for column in ['d1', 'd2', 'd3']:
    X_train[column] = le_features.fit_transform(X_train[column])
    X_test[column] = le_features.transform(X_test[column])
# Train a classifier (RandomForest is a good starting point)
model = RandomForestClassifier(random_state=12) # You can try other models (e.g., Gradient Boosting)
model.fit(X_train, y_train)
# Make predictions on the test set
y_pred = model.predict(X_test)

# Evaluate the model (convert predictions back to original labels for reporting)
#y_pred_labels = le.inverse_transform(y_pred)
print("Classification Report:\n", classification_report(y_test, y_pred))

# Evaluate the model
#print(classification_report(y_test, y_pred))

# Feature Importance (to see which lags are most influential)
feature_importances = model.feature_importances_
print("Feature Importances:", feature_importances)

Classification Report:
               precision    recall  f1-score   support

           0       0.41      0.28      0.33        40
           1       0.69      0.80      0.74        81

    accuracy                           0.63       121
   macro avg       0.55      0.54      0.54       121
weighted avg       0.60      0.63      0.61       121

Feature Importances: [0.17870979 0.1634829  0.17762888 0.24458361 0.23559483]
